### Oracle 서버와 파이썬 연동
- uv pip install oracledb
- uv pip install python-dotenv

In [19]:
import oracledb
from dotenv import load_dotenv      #python-dotenv 설치해서 가능한 것
import os

conn = oracledb.connect(user="python_user", password = "54321", dsn = "localhost/xe")
cursor = conn.cursor()

cursor.execute("select * from post")

for row in cursor.fetchall():
    print(row)

conn.close()

In [16]:
# .env 파일 읽어오기
load_dotenv()

#  오라클 비밀번호 가져오기
password = os.getenv("ORACLE_PASSWORD")

In [ ]:
# sql = "insert into mart_member(member_id, name, password) values(:1, :2, :3)"
sql = "insert into mart_member(member_id, name, password) values(:member_id, :name, :password)"

with oracledb.connect(user="python_user", password = password, dsn = "localhost/xe") as conn:
    with conn.cursor() as cursor:
        cursor.execute(sql, ('hong123', '홍길동', 'hong12300'))
        conn.commit()

In [26]:
with oracledb.connect(user="python_user", password = password, dsn = "localhost/xe") as conn:
    with conn.cursor() as cursor:
        cursor.execute("select * from mart_member")
        for row in cursor.fetchall():
            print(row)

('hong123', 'hong12300', '홍길동', None, None, None)
('kim123', 'kim123000', '김길동', None, None, None)
('cho123', 'choi123456', '조미연', None, None, None)


In [24]:
sql = "insert into mart_member(member_id, name, password) values(:member_id, :name, :password)"

with oracledb.connect(user="python_user", password = password, dsn = "localhost/xe") as conn:
    with conn.cursor() as cursor:
        cursor.execute(sql, ('kim123', '김길동', 'kim123000'))
        conn.commit()

In [ ]:
# 딕셔너리(객체)로 줄 때 → 키 이름이 바인드 변수 이름과 정확히 일치해야 함
# values(:1, :2) 이렇게 주면 딕셔너리(객체)로 데이터를 선언못함. 

sql = "insert into mart_member(member_id, name, password) values(:member_id, :name, :password)"

with oracledb.connect(user="python_user", password = password, dsn = "localhost/xe") as conn:
    with conn.cursor() as cursor:
        data = {
            "member_id" : "cho123", 
            "name" : "조미연", 
            "password" : "choi123456"
        }
        cursor.execute(sql, data)
        conn.commit()

In [27]:
sql = "insert into category(category_id, category_name) values(:1, :2)"

with oracledb.connect(user="python_user", password = password, dsn = "localhost/xe") as conn:
    with conn.cursor() as cursor:
        cursor.execute(sql, (1101, '경제/경영'))
        conn.commit()

In [28]:
with oracledb.connect(user="python_user", password = password, dsn = "localhost/xe") as conn:
    with conn.cursor() as cursor:
        cursor.execute("select * from category")
        for row in cursor.fetchall():
            print(row)

(1101, '경제/경영')


In [31]:
sql = "insert into category(category_id, category_name) values(:1, :2)"

data = [
    [1102, '요리'],
    [1103, '예술'],
    [1104, '정치']
]

with oracledb.connect(user="python_user", password = password, dsn = "localhost/xe") as conn:
    with conn.cursor() as cursor:
        cursor.executemany(sql, data)
        conn.commit()

In [32]:
# 사회 => 자기계발

sql = "update category set category_name=:1 where category_id = :2"

with oracledb.connect(user="python_user", password = password, dsn = "localhost/xe") as conn:
    with conn.cursor() as cursor:
        cursor.execute(sql, ("자기계발", 1104))
        conn.commit()

In [33]:
# 삭제

sql = "delete from category where category_id = :1"

with oracledb.connect(user="python_user", password = password, dsn = "localhost/xe") as conn:
    with conn.cursor() as cursor:
        cursor.execute(sql, (1103,))
        conn.commit()

In [44]:
from faker import Faker

fake = Faker(locale="ko_KR")
Faker.seed(4321)

print(fake.name())
print(fake.email())
print(fake.address())
print(fake.ssn())
print(fake.phone_number())

이도윤
sunoggim@example.net
대구광역시 영등포구 서초중앙거리 238-52
670227-2018273
070-4938-5639


In [ ]:
from faker.providers import internet

fake.add_provider(internet)

for i in range(10):
    print(fake.ipv4_private())

In [40]:
fake.date_of_birth(minimum_age=20, maximum_age=30)
fake.date_of_birth(minimum_age=20, maximum_age=30).strftime("%Y-%m-%d")

'2003-11-03'

In [41]:
fake.date_time_between(start_date="-3y", end_date="now")

datetime.datetime(2026, 1, 4, 7, 32, 55)

In [ ]:
# mart_member에 50명의 임의의 데이터를 삽입하고 싶음
# name, birth, phone, joined_at

# 50명의 데이터를 리스트에 추가

faker = Faker(locale="ko_KR")
# Faker.seed(4321);

data = []

# i 값을 사용하지 않을때는 _ 로 사용하기도 함
for i in range(50):
    id = i
    name = faker.name()
    birth = faker.date_of_birth(minimum_age=19, maximum_age=70)   
    phone = faker.phone_number()
    joined_at = faker.date_between(start_date="-3y", end_date="now")  

    data.append([id, name, birth, phone, joined_at])

print(data)

sql = "insert into member(member_id, name, birth, phone, joined_at) values(:1, :2, :3, :4, :5)"

with oracledb.connect(user="python_user", password = password, dsn = "localhost/xe") as conn:
    with conn.cursor() as cursor:
        cursor.executemany(sql, data)
        conn.commit()
# insert로 실행

[[0, '김은경', datetime.date(1968, 4, 29), '031-925-0761', datetime.datetime(2024, 7, 15, 12, 41, 43)], [1, '황춘자', datetime.date(1975, 12, 12), '042-500-6481', datetime.datetime(2025, 3, 24, 4, 32, 56)], [2, '김서준', datetime.date(1980, 11, 24), '044-356-8262', datetime.datetime(2024, 4, 18, 8, 54, 29)], [3, '김현우', datetime.date(1992, 6, 26), '031-646-7057', datetime.datetime(2023, 12, 20, 3, 21, 14)], [4, '배광수', datetime.date(1965, 9, 9), '011-332-6122', datetime.datetime(2026, 8, 11, 0, 43, 28)], [5, '이서윤', datetime.date(1989, 1, 25), '070-6239-2397', datetime.datetime(2026, 4, 5, 9, 46, 11)], [6, '한현정', datetime.date(1986, 8, 22), '032-868-8532', datetime.datetime(2025, 2, 27, 17, 51, 5)], [7, '이수빈', datetime.date(1993, 4, 5), '019-899-5727', datetime.datetime(2025, 3, 21, 1, 16, 50)], [8, '김성수', datetime.date(1960, 11, 4), '02-9565-5967', datetime.datetime(2023, 11, 19, 6, 29, 59)], [9, '한혜진', datetime.date(2000, 3, 8), '055-795-9601', datetime.datetime(2024, 1, 22, 17, 16, 21)], [10, '